# This notebook was used to evaluate the inference results of Transformer-40k tested mouse genome
I tested the model on two versions of ENSEMBL, v114 and v87. v114 was the most recent ENSEMBL mouse annotation, so I tested on this data first. I realized that if I am evaluating the performance compared to the human dataset, it may make sense to compare them on the same ENSEMBL version. So the v87 test was done as this was the same ENSEMBL version as the human test from the paper. 

In [ ]:
import numpy as np
import sys
import time
import h5py
from tqdm import tqdm

import numpy as np
import re
from math import ceil
from sklearn.metrics import average_precision_score
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import pickle
#import pickle5 as pickle

from sklearn.model_selection import train_test_split

from scipy.sparse import load_npz
from glob import glob

from transformers import get_constant_schedule_with_warmup
from sklearn.metrics import precision_score,recall_score,accuracy_score
import copy

from src.train import trainModel
#from src.dataloader import getData,spliceDataset,h5pyDataset,collate_fn
from src.dataloader import getData,spliceDataset,h5pyDataset,getDataPointList,getDataPointListFull,DataPointFull
from src.weight_init import keras_init
from src.losses import categorical_crossentropy_2d
from src.model import SpliceFormer
from src.evaluation_metrics import print_topl_statistics,cross_entropy_2d
from src.gpu_metrics import run_bootstrap, calculate_ap, calculate_topk
import os

First checking the mouse results from ENSEMBL v114 (newest version)

In [3]:
# Needed the skiprows=215000 due to a mistake when performing inference in compute canada. The first 215000 rows are an artifact from an earlier experiment
m114 = pd.read_csv('../Data/Mouse/transformer_40k_test_mouse_ensembl_predictions_070725.csv.gz', skiprows=215000, header=0)
m114

,Y_true_acceptor,Y_pred_acceptor,Y_true_donor,Y_pred_donor
0,0.0,2.190716e-07,0.0,3.742390e-07
1,0.0,8.120200e-08,0.0,9.200427e-07
2,0.0,7.164110e-08,0.0,1.128564e-06
3,0.0,7.066605e-08,0.0,4.843365e-06
4,0.0,5.708911e-05,0.0,5.471461e-06
...,...,...,...,...
471449995,0.0,4.048695e-06,0.0,7.481128e-06
471449996,0.0,4.202282e-06,0.0,7.584809e-06
471449997,0.0,4.255830e-06,0.0,7.886042e-06
471449998,0.0,4.165138e-06,0.0,7.935830e-06


In [5]:
device = torch.device("cpu")
Y_true_acceptor = torch.as_tensor(m114['Y_true_acceptor'].values, dtype=torch.int8).to(device)
Y_pred_acceptor = torch.as_tensor(m114['Y_pred_acceptor'].values, dtype=torch.float32).to(device)
Y_true_donor = torch.as_tensor(m114['Y_true_donor'].values, dtype=torch.int8).to(device)
Y_pred_donor = torch.as_tensor(m114['Y_pred_donor'].values, dtype=torch.float32).to(device)


# Compute scores
ap_score = calculate_ap(Y_true_acceptor, Y_pred_acceptor, Y_true_donor, Y_pred_donor, device, device)
topk_score = calculate_topk(Y_true_acceptor, Y_pred_acceptor, Y_true_donor, Y_pred_donor)

print(f"Average Precision: {ap_score:.4f}")
print(f"Top-k Accuracy: {topk_score:.4f}")

Average Precision: 0.9796
Top-k Accuracy: 0.9488


In [11]:
from sklearn.metrics import average_precision_score
a = average_precision_score(Y_true_acceptor,Y_pred_acceptor)
b = average_precision_score(Y_true_donor,Y_pred_donor)
print(a,b)


0.9773672218680703 0.981888712911462
1.4683115783238012


In [4]:
# Checking performance at a high threshold of 0.8. Basically wanting to see performance if we only consider scores above 0.8 as correctly identified splice sites
threshold = 0.8
detected_acceptors = np.sum((m114.Y_true_acceptor == 1) & (m114.Y_pred_acceptor >= threshold))
missed_acceptors   = np.sum((m114.Y_true_acceptor == 1) & (m114.Y_pred_acceptor < threshold))
total_acceptors    = np.sum(m114.Y_true_acceptor == 1)

print(f"Detected acceptor sites: {detected_acceptors} / {total_acceptors}")
print(f"Missed acceptor sites:   {missed_acceptors}")
print('Ratio:', detected_acceptors / total_acceptors)

detected_donor = np.sum((m114.Y_true_donor == 1) & (m114.Y_pred_donor >= threshold))
missed_donor  = np.sum((m114.Y_true_donor == 1) & (m114.Y_pred_donor < threshold))
total_donor    = np.sum(m114.Y_true_donor == 1)

print(f"Detected donor sites: {detected_donor} / {total_donor}")
print(f"Missed donor sites:   {missed_donor}")
print('Ratio:', detected_donor / total_donor)

Detected acceptor sites: 73692 / 85481
Missed acceptor sites:   11789
Ratio: 0.8620863115780115
Detected donor sites: 74583 / 85481
Missed donor sites:   10898
Ratio: 0.8725096805137984


Now comparing to the human results from ENSEMBL v87 (same version as paper). Top-k and PR-AUC already calculated in transformer_vs_spliceai.ipynb notebook and have values:
Average Precision: 0.9687,
Top-k Accuracy: 0.9433

In [5]:
# Original human results
h87 = pd.read_csv('../Data/transformer_40k_test_ensembl_predictions_300625.csv.gz')
h87

,Y_true_acceptor,Y_pred_acceptor,Y_true_donor,Y_pred_donor
0,0.0,2.336246e-07,0.0,0.000002
1,0.0,2.312783e-07,0.0,0.000003
2,0.0,4.485540e-08,0.0,0.000007
3,0.0,2.567490e-07,0.0,0.000005
4,0.0,1.211019e-07,0.0,0.000001
...,...,...,...,...
664939995,0.0,3.311274e-06,0.0,0.000008
664939996,0.0,3.702454e-06,0.0,0.000008
664939997,0.0,3.740225e-06,0.0,0.000008
664939998,0.0,3.670163e-06,0.0,0.000007


In [6]:
# Using the same 0.8 threshold as before
detected_acceptors = np.sum((h87.Y_true_acceptor == 1) & (h87.Y_pred_acceptor >= threshold))
missed_acceptors   = np.sum((h87.Y_true_acceptor == 1) & (h87.Y_pred_acceptor < threshold))
total_acceptors    = np.sum(h87.Y_true_acceptor == 1)

print(f"Detected acceptor sites: {detected_acceptors} / {total_acceptors}")
print(f"Missed acceptor sites:   {missed_acceptors}")
print('Ratio:', detected_acceptors / total_acceptors)

detected_donor = np.sum((h87.Y_true_donor == 1) & (h87.Y_pred_donor >= threshold))
missed_donor  = np.sum((h87.Y_true_donor == 1) & (h87.Y_pred_donor < threshold))
total_donor    = np.sum(h87.Y_true_donor == 1)

print(f"Detected donor sites: {detected_donor} / {total_donor}")
print(f"Missed donor sites:   {missed_donor}")
print('Ratio:', detected_donor / total_donor)

Detected acceptor sites: 78493 / 89712
Missed acceptor sites:   11219
Ratio: 0.8749442660959514
Detected donor sites: 79746 / 89712
Missed donor sites:   9966
Ratio: 0.8889111824505083


In [7]:
def get_top_k(y_true, y_pred):
    idx_true = np.nonzero(y_true == 1)[0]
    argsorted_y_pred = np.argsort(y_pred)
    sorted_y_pred = np.sort(y_pred)

    topkl_accuracy = []
    threshold = []
    top_length = 1
    idx_pred = argsorted_y_pred[-int(top_length*len(idx_true)):]
    correct = np.size(np.intersect1d(idx_true, idx_pred))
    total = float(min(len(idx_pred), len(idx_true)))
    if top_length == 1:
        correct_1 = correct
        total_1 = total
    topkl_accuracy += [ correct/ total]
    threshold += [sorted_y_pred[-int(top_length*len(idx_true))]]
    return topkl_accuracy,threshold

Just checking top-k for each donor and acceptor and their respective thresholds

In [9]:
h87_acc_topk,h87_acc_threshold = get_top_k(h87.Y_true_acceptor.values, h87.Y_pred_acceptor.values)
h87_donor_topk,h87_donor_threshold = get_top_k(h87.Y_true_donor.values, h87.Y_pred_donor.values)
print('Acceptor:',h87_acc_topk,h87_acc_threshold)
print('Donor:',h87_donor_topk,h87_donor_threshold)
print((h87_acc_topk[0]+h87_donor_topk[0])/2)

Acceptor: [0.9408663278045301] [np.float64(0.45556268)]
Donor: [0.9457040306759408] [np.float64(0.4889393)]
0.9432851792402355


In [10]:
m114_acc_topk,m114_acc_threshold = get_top_k(m114.Y_true_acceptor.values, m114.Y_pred_acceptor.values)
m114_donor_topk,m114_donor_threshold = get_top_k(m114.Y_true_donor.values, m114.Y_pred_donor.values)
print('Acceptor:',m114_acc_topk,m114_acc_threshold)
print('Donor:',m114_donor_topk,m114_donor_threshold)
print((m114_acc_topk[0]+m114_donor_topk[0])/2)

Acceptor: [0.946327254009663] [np.float64(0.3904859)]
Donor: [0.9512640235841883] [np.float64(0.4115452)]
0.9487956387969256


Now checking results for mouse ENSEMBL v87. I thought maybe it would be good to test the same ENSEMBL version for mouse as was done with human for consistency.

In [11]:
m87 = pd.read_csv('../Data/Mouse/v87/transformer_40k_test_mouse_ensembl_predictions_090725.csv.gz')
m87

,Y_true_acceptor,Y_pred_acceptor,Y_true_donor,Y_pred_donor
0,0.0,1.983286e-06,0.0,0.000006
1,0.0,7.572056e-07,0.0,0.000004
2,0.0,2.277105e-07,0.0,0.000011
3,0.0,5.697826e-07,0.0,0.000003
4,0.0,6.817683e-07,0.0,0.000002
...,...,...,...,...
455524995,0.0,2.839530e-05,0.0,0.000056
455524996,0.0,2.839530e-05,0.0,0.000056
455524997,0.0,2.839530e-05,0.0,0.000056
455524998,0.0,2.839530e-05,0.0,0.000056


In [13]:
device = torch.device("cpu")
Y_true_acceptor = torch.as_tensor(m87['Y_true_acceptor'].values, dtype=torch.int8).to(device)
Y_pred_acceptor = torch.as_tensor(m87['Y_pred_acceptor'].values, dtype=torch.float32).to(device)
Y_true_donor = torch.as_tensor(m87['Y_true_donor'].values, dtype=torch.int8).to(device)
Y_pred_donor = torch.as_tensor(m87['Y_pred_donor'].values, dtype=torch.float32).to(device)


# Compute scores
ap_score = calculate_ap(Y_true_acceptor, Y_pred_acceptor, Y_true_donor, Y_pred_donor, device, device)
topk_score = calculate_topk(Y_true_acceptor, Y_pred_acceptor, Y_true_donor, Y_pred_donor)

print(f"Average Precision: {ap_score:.4f}")
print(f"Top-k Accuracy: {topk_score:.4f}")

Average Precision: 0.9787
Top-k Accuracy: 0.9480


: 

In [12]:
threshold = 0.8
detected_acceptors = np.sum((m87.Y_true_acceptor == 1) & (m87.Y_pred_acceptor >= threshold))
missed_acceptors   = np.sum((m87.Y_true_acceptor == 1) & (m87.Y_pred_acceptor < threshold))
total_acceptors    = np.sum(m87.Y_true_acceptor == 1)

print(f"Detected acceptor sites: {detected_acceptors} / {total_acceptors}")
print(f"Missed acceptor sites:   {missed_acceptors}")
print('Ratio:', detected_acceptors / total_acceptors)

detected_donor = np.sum((m87.Y_true_donor == 1) & (m87.Y_pred_donor >= threshold))
missed_donor  = np.sum((m87.Y_true_donor == 1) & (m87.Y_pred_donor < threshold))
total_donor    = np.sum(m87.Y_true_donor == 1)

print(f"Detected donor sites: {detected_donor} / {total_donor}")
print(f"Missed donor sites:   {missed_donor}")
print('Ratio:', detected_donor / total_donor)

Detected acceptor sites: 70309 / 81671
Missed acceptor sites:   11362
Ratio: 0.8608808512201394
Detected donor sites: 71176 / 81671
Missed donor sites:   10495
Ratio: 0.8714966144653549
